# EnergyAI – komplett analys

**Frågeställning:** Kan maskininlärning och deep learning användas för att förutsäga energiförbrukning och identifiera anomalier?

Denna notebook täcker datautvinning, EDA, klassisk ML, deep learning och anomalidetektion. Dataströmsdelen finns som separata Kafka-program.

In [ ]:
# Installera vid behov
# %pip install pandas numpy matplotlib seaborn scikit-learn ucimlrepo tensorflow joblib

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import tensorflow as tf
from tensorflow import keras
from sklearn.neural_network import MLPRegressor

sns.set_theme(style="whitegrid")
dataset = fetch_ucirepo(id=374)
df = pd.concat([dataset.data.features.copy(), dataset.data.targets.copy()], axis=1)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df["hour"] = df.date.dt.hour
df["dayofweek"] = df.date.dt.dayofweek
df["month"] = df.date.dt.month
df["is_weekend"] = (df.dayofweek >= 5).astype(int)
df["lag_1"] = df.Appliances.shift(1)
df["lag_6"] = df.Appliances.shift(6)
df["lag_144"] = df.Appliances.shift(144)
df = df.dropna().reset_index(drop=True)
df.head()

In [ ]:
print(df.shape)
display(df.describe().T)
print("Missing values:", int(df.isna().sum().sum()))

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(df.date, df.Appliances, linewidth=.7)
plt.title("Energy consumption over time")
plt.xlabel("Date"); plt.ylabel("Wh"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9,7))
cols=["Appliances","T_out","RH_out","Windspeed","hour","dayofweek","lag_1","lag_6","lag_144"]
sns.heatmap(df[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation matrix"); plt.tight_layout(); plt.show()

## Modellering – tidsbaserad validering

In [ ]:
features=["lights","T1","RH_1","T2","RH_2","T3","RH_3","T4","RH_4","T5","RH_5",
"T6","RH_6","T7","RH_7","T8","RH_8","T9","RH_9","T_out","Press_mm_hg","RH_out",
"Windspeed","Visibility","Tdewpoint","hour","dayofweek","month","is_weekend","lag_1","lag_6","lag_144"]
X,y=df[features],df.Appliances
split=int(len(df)*.8)
Xtr,Xte=X.iloc[:split],X.iloc[split:]; ytr,yte=y.iloc[:split],y.iloc[split:]

models={
"Linear Regression":Pipeline([("scaler",StandardScaler()),("model",LinearRegression())]),
"Decision Tree":DecisionTreeRegressor(max_depth=12,random_state=42),
"Random Forest":RandomForestRegressor(n_estimators=200,max_depth=18,random_state=42,n_jobs=-1)
}
results=[]; preds={}
for name,m in models.items():
    m.fit(Xtr,ytr); p=m.predict(Xte); preds[name]=p
    results.append([name,mean_absolute_error(yte,p),np.sqrt(mean_squared_error(yte,p)),r2_score(yte,p)])
results_df=pd.DataFrame(results,columns=["Model","MAE","RMSE","R2"]).sort_values("RMSE")
results_df

In [ ]:
best=results_df.iloc[0]["Model"]
plt.figure(figsize=(14,5))
n=min(1000,len(yte))
plt.plot(yte.iloc[:n].values,label="Actual")
plt.plot(preds[best][:n],label="Predicted")
plt.title(f"Actual vs predicted – {best}")
plt.legend(); plt.tight_layout(); plt.show()

## Deep Learning – MLP

In [ ]:
# MLPRegressor är ett lättare sätt att demonstrera ett neuralt nätverk
mlp=Pipeline([
    ("scaler",StandardScaler()),
    ("model",MLPRegressor(hidden_layer_sizes=(64,32),activation="relu",
                          max_iter=100,early_stopping=True,random_state=42))
])
mlp.fit(Xtr,ytr)
p_mlp=mlp.predict(Xte)
mlp_result=[mean_absolute_error(yte,p_mlp),np.sqrt(mean_squared_error(yte,p_mlp)),r2_score(yte,p_mlp)]
pd.DataFrame([mlp_result],columns=["MAE","RMSE","R2"],index=["Neural Network"])

## Anomalidetektion

In [ ]:
af=df[["Appliances","T_out","RH_out","Windspeed","hour","dayofweek"]]
iso=IsolationForest(n_estimators=200,contamination=.02,random_state=42)
df["is_anomaly"]=iso.fit_predict(af)==-1
print("Flagged anomalies:", int(df.is_anomaly.sum()))
display(df[df.is_anomaly][["date","Appliances","T_out","RH_out"]].head(20))

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(df.date,df.Appliances,linewidth=.6,label="Normal")
a=df[df.is_anomaly]
plt.scatter(a.date,a.Appliances,s=12,label="Anomaly")
plt.title("Anomaly detection"); plt.ylabel("Wh"); plt.legend(); plt.tight_layout(); plt.show()

## Slutsats – fyll i efter körning

Besvara:
- Vilken modell presterade bäst och enligt vilket mått?
- Hur stor var skillnaden mot baseline?
- Hur fungerade det neurala nätverket jämfört med klassiska modeller?
- Vad innebär anomalierna?
- Vilka begränsningar finns?
- Hur skulle Kafka-delen förändra systemets arkitektur?
- Hur skulle lösningen skalas till fler byggnader/sensorer?

**Källor:** UCI Machine Learning Repository, Appliances Energy Prediction (Candanedo, 2017).